In [ ]:
# --- CELL 1: Install Optimized Libraries ---
# 1. Fetch pre-compiled xformers wheel from PyTorch index (takes 5 seconds)
!pip install --no-deps xformers --index-url https://download.pytorch.org/whl/cu121

# 2. Install unsloth & dependencies without compiling
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git"

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 84.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 1.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 30.4 MB/s eta 0:00:0000:0100:01
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xnwz3nzc/unsloth_7798d8b1584a47f581b6ca209d805e83
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xnwz3nzc/unsloth_7798d8b1584a47f581b6ca209d805e83
  Resolved https://github.com/unslothai/unsloth.git to commit 6e21a68ebdfea3d8f3dd378e5f895841f0ddd2bd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.5 MB/s 

In [ ]:
# --- CELL 2: Load Model in 4-bit Precision ---
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.23: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 5.070 GiB
no_split classes   : ['Qwen2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 10.260 GiB requested
  cuda:0  budget  12.95 GiB  weights  2.643 GiB  free 10.309 GiB  reserve 10.260 GiB
  cuda:1  budget  13.00 GiB  weights  2.426 GiB  free 10.570 GiB  reserve 10.103 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.440 -> 12.996 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
# --- CELL 3: Configure LoRA Parameter Efficient Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.8.23 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".jsonl") or file.endswith(".json"):
            print(os.path.join(root, file))

/kaggle/input/datasets/ahmedsamy2003/train-data-jsonl/train_data.jsonl


In [ ]:
# --- CELL 4: Load Synthetic Dataset ---
from datasets import load_dataset
# Update dataset path to match your uploaded Kaggle dataset
dataset = load_dataset("json",
    data_files="/kaggle/input/datasets/ahmedsamy2003/train-data-jsonl/train_data.jsonl",
    split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
print(dataset[0].keys())

dict_keys(['messages'])


In [ ]:
# --- CELL 5: Train Model with SFTTrainer ---
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Define the chat template formatting function
def format_prompts(examples):
    # Unsloth/Hugging Face batch processing or single sample check
    if isinstance(examples["messages"][0], list):
        texts = [
            tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
            for msg in examples["messages"]
        ]
    else:
        texts = [tokenizer.apply_chat_template(examples["messages"], tokenize=False, add_generation_prompt=False)]
    return texts

# 2. Initialize SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    formatting_func = format_prompts,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/907 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 907 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.636860
2,3.805254
3,3.632443
4,3.276060
5,3.106787
6,2.686956
7,2.537882
8,2.136517
9,1.804923
10,1.679637


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
# Save light adapters (~150MB) directly
model.save_pretrained("qwen2.5_gym_coach_lora")
tokenizer.save_pretrained("qwen2.5_gym_coach_lora")

# Create zip file
shutil.make_archive("qwen2.5_gym_coach_lora", 'zip', "qwen2.5_gym_coach_lora")

print("[PASS] Saved and zipped adapters successfully without filling disk space.")

[PASS] Saved and zipped adapters successfully without filling disk space.


In [ ]:
import os
import shutil
from IPython.display import FileLink

# Ensure working directory is /kaggle/working
os.chdir('/kaggle/working')

# 1. Zip the adapters directory if it isn't zipped yet
if not os.path.exists("qwen2.5_gym_coach_lora.zip"):
    shutil.make_archive("qwen2.5_gym_coach_lora", 'zip', "qwen2.5_gym_coach_lora")

# 2. Display a direct download link in the notebook output
FileLink(r'qwen2.5_gym_coach_lora.zip')

/kaggle/working/qwen2.5_gym_coach_lora.zip

In [ ]:
FastLanguageModel.for_inference(model)

test_queries = [
    "My lower back feels completely compressed during conventional deadlifts.",
    "My bench press has been stuck at 85kg for 4 weeks in a row.",
    "I'm at a hotel gym with only 15kg dumbbells. Give me a shoulder workout.",
    "Where should my elbows be positioned during a flat bench press?"
]

print("=== RUNNING INFERENCE BENCHMARK ===")
for q in test_queries:
    messages = [
        {"role": "system", "content": "You are an elite Gym Coach AI. You manage user programming, injuries, and progressive overload. You have access to tools: 'search_medical_db', 'search_exercise_catalog', 'generate_workout'. If an injury or plateau is present, output strict JSON tool calls. If direct form advice, output coaching text."},
        {"role": "user", "content": q}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True,
        add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=150,
        use_cache=True)
    decoded = tokenizer.batch_decode(outputs)
    print(f"\n[QUERY]:\n{q}\n[RESPONSE]:\n{decoded[0].split('<|im_start|>assistant')[-1]}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RUNNING INFERENCE BENCHMARK ===


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[QUERY]:
My lower back feels completely compressed during conventional deadlifts.
[RESPONSE]:

{"thought": "Lumbar compression during conventional deadlifts suggests a potential lumbar spine issue, like lumbar spinal stenosis or disc herniation. I will search the medical database for lumbar spine compression during deadlifts.", "tool_call": {"name": "search_medical_db", "arguments": {"query": "lumbar compression conventional deadlift spine stenosis"}}}<|im_end|>


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[QUERY]:
My bench press has been stuck at 85kg for 4 weeks in a row.
[RESPONSE]:

{"thought": "Stuck at 85kg on bench press for 4 weeks indicates a true plateau. I will search the database for bench press plateaus and volume overreaching.", "tool_call": {"name": "search_medical_db", "arguments": {"query": "bench press plateau 85kg 4 weeks"}}}<|im_end|>


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[QUERY]:
I'm at a hotel gym with only 15kg dumbbells. Give me a shoulder workout.
[RESPONSE]:

{"thought": "The lifter has 15kg dumbbells in a hotel gym. I need to search the exercise catalog for low-weight shoulder exercises.", "tool_call": {"name": "search_exercise_catalog", "arguments": {"query": "shoulder workout using 15kg dumbbells"}}}<|im_end|>

[QUERY]:
Where should my elbows be positioned during a flat bench press?
[RESPONSE]:

Your elbows should be at about a 45-degree angle to the ground when performing a flat bench press. This helps keep your shoulders in a safer, more stable position, reducing the risk of shoulder impingement and rotator cuff strain.<|im_end|>
